# Notebook 3 — Evaluation

Run on **Google Colab** (T4 GPU) — same session as notebook 02, or a fresh one with the adapter uploaded.

Compares base Llama 3.2 3B vs fine-tuned model on the test set.

**Metrics:**
1. JSON validity rate
2. Least-privilege compliance
3. NIST mapping recall
4. ROUGE-L

In [1]:
!pip install unsloth evaluate rouge_score -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.1 MB/s eta 0:00:00
   ━━━━━

## 0. Load adapter and test data from Google Drive

In [2]:
import os, shutil, json
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/iam-policy-llm'

# Copy adapter
adapter_dst = '/content/iam-policy-adapter'
if os.path.exists(adapter_dst):
    shutil.rmtree(adapter_dst)
shutil.copytree(f'{DRIVE}/iam-policy-adapter', adapter_dst)

# Copy test set
os.makedirs('data/processed', exist_ok=True)
shutil.copy(f'{DRIVE}/data/processed/test.jsonl', 'data/processed/test.jsonl')

print('Adapter:', os.listdir(adapter_dst))
print('Test set copied')

Mounted at /content/drive
Adapter: ['adapter_config.json', 'chat_template.jinja', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'tokenizer.json']
Test set copied


## 1. Load test set

In [3]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def extract_actions(policy):
    actions = set()
    for stmt in policy.get('Statement', []):
        if not isinstance(stmt, dict):
            continue
        raw = stmt.get('Action', [])
        if isinstance(raw, str):
            actions.add(raw)
        else:
            actions.update(raw)
    return actions

test_data = load_jsonl('data/processed/test.jsonl')
print(f'{len(test_data)} test examples')

51 test examples


## 2. Load model

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = '/content/iam-policy-adapter',
    max_seq_length = 2048,
    load_in_4bit = True,
)
print('Model loaded')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.5 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Model loaded


## 3. Run inference — fine-tuned and base model

In [5]:
import re

def generate(requirement, mdl, tok):
    prompt = (
        "### Instruction:\nGenerate an AWS IAM policy for the following requirement\n\n"
        f"### Input:\n{requirement}\n\n"
        "### Response:\n"
    )
    inputs = tok(prompt, return_tensors='pt').to(mdl.device)
    outputs = mdl.generate(**inputs, max_new_tokens=600, temperature=0.1, do_sample=False)
    decoded = tok.decode(outputs[0], skip_special_tokens=True)
    text = decoded.split('### Response:')[-1].strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except json.JSONDecodeError:
                pass
        return {'parse_error': True, 'raw_output': text}


# Fine-tuned inference
FastLanguageModel.for_inference(model)
ft_outputs = []
for i, ex in enumerate(test_data):
    ft_outputs.append(generate(ex['input'], model, tokenizer))
    if (i + 1) % 10 == 0:
        print(f'  Fine-tuned: {i+1}/{len(test_data)}')
print(f'Fine-tuned done — {len(ft_outputs)} examples')

# Base model inference — disable the LoRA adapter
base_outputs = []
with model.disable_adapter():
    for i, ex in enumerate(test_data):
        base_outputs.append(generate(ex['input'], model, tokenizer))
        if (i + 1) % 10 == 0:
            print(f'  Base: {i+1}/{len(test_data)}')
print(f'Base done — {len(base_outputs)} examples')

Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

  Fine-tuned: 10/51


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `tr

  Fine-tuned: 20/51


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  Base: 30/51


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  Base: 40/51


Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  Base: 50/51
Base done — 51 examples


## 4. Compute metrics

In [1]:
from evaluate import load as load_metric
rouge = load_metric('rouge')

def json_validity_rate(outputs):
    valid = sum(1 for o in outputs if not o.get('parse_error'))
    return valid / len(outputs)

def least_privilege_rate(outputs, ground_truths):
    results = []
    for o, gt in zip(outputs, ground_truths):
        if o.get('parse_error'):
            results.append(False)
            continue
        try:
            gt_actions = extract_actions(gt['output']['policy'])
            pred_actions = extract_actions(o.get('policy', {}))
            results.append(len(pred_actions - gt_actions) == 0)
        except Exception:
            results.append(False)
    return sum(results) / len(results)

def nist_recall(outputs, ground_truths):
    scores = []
    for o, gt in zip(outputs, ground_truths):
        gt_controls = set(gt['output'].get('nist_controls', []))
        if not gt_controls:
            continue
        if o.get('parse_error'):
            scores.append(0.0)
            continue
        pred_controls = set(o.get('nist_controls', []))
        scores.append(len(pred_controls & gt_controls) / len(gt_controls))
    return sum(scores) / len(scores) if scores else 0.0

def rouge_l(outputs, ground_truths):
    preds = [json.dumps(o) for o in outputs]
    refs  = [json.dumps(ex['output']) for ex in ground_truths]
    return rouge.compute(predictions=preds, references=refs)['rougeL']


base_json  = json_validity_rate(base_outputs)
ft_json    = json_validity_rate(ft_outputs)

base_lp    = least_privilege_rate(base_outputs, test_data)
ft_lp      = least_privilege_rate(ft_outputs, test_data)

base_nist  = nist_recall(base_outputs, test_data)
ft_nist    = nist_recall(ft_outputs, test_data)

base_rouge = rouge_l(base_outputs, test_data)
ft_rouge   = rouge_l(ft_outputs, test_data)

print('Metrics computed')

ModuleNotFoundError: No module named 'evaluate'

## 5. Results

In [7]:
import pandas as pd

summary = pd.DataFrame({
    'Metric': ['JSON validity', 'Least-privilege compliance', 'NIST mapping recall', 'ROUGE-L'],
    'Base Llama 3.2 3B': [
        f'{base_json:.1%}', f'{base_lp:.1%}', f'{base_nist:.1%}', f'{base_rouge:.3f}'
    ],
    'Fine-tuned (QLoRA)': [
        f'{ft_json:.1%}', f'{ft_lp:.1%}', f'{ft_nist:.1%}', f'{ft_rouge:.3f}'
    ],
})
summary

,Metric,Base Llama 3.2 3B,Fine-tuned (QLoRA)
0,JSON validity,78.4%,96.1%
1,Least-privilege compliance,78.4%,21.6%
2,NIST mapping recall,0.0%,53.3%
3,ROUGE-L,0.326,0.589


## 6. Save results to Drive

In [8]:
results = {
    'base':       {'json_validity': base_json,  'least_privilege': base_lp,  'nist_recall': base_nist,  'rouge_l': base_rouge},
    'finetuned':  {'json_validity': ft_json,    'least_privilege': ft_lp,    'nist_recall': ft_nist,    'rouge_l': ft_rouge},
}

out_path = f'{DRIVE}/eval_results.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))

Results saved to /content/drive/MyDrive/iam-policy-llm/eval_results.json
{
  "base": {
    "json_validity": 0.7843137254901961,
    "least_privilege": 0.7843137254901961,
    "nist_recall": 0.0,
    "rouge_l": 0.32578941853339327
  },
  "finetuned": {
    "json_validity": 0.9607843137254902,
    "least_privilege": 0.21568627450980393,
    "nist_recall": 0.5326797385620915,
    "rouge_l": 0.5891755212664052
  }
}
